# 07 - Run the Streamlit app on a Kaggle GPU
Runs `app.py` on Kaggle's T4 and gives you a public link (Cloudflare tunnel) to open in your browser.

**Setup**
1. Input: the `mural-final` project dataset (and `MuralDH` if you want to pick test images).
2. Accelerator **GPU T4**, Internet **ON**.
3. Run the cells **one by one in the interactive editor** (not *Save Version*). Keep this tab open while you use the app.


In [ ]:
!pip install -q streamlit streamlit-image-coordinates lpips peft diffusers transformers accelerate safetensors scikit-image opencv-python-headless


In [ ]:
import os, sys, glob, shutil, subprocess, time, re, urllib.request
import torch
assert torch.cuda.is_available(), "Turn on GPU T4 (Settings -> Accelerator)"

# ---- copy the project to a writable folder
cands = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/app.py", recursive=True)
         if os.path.isdir(os.path.join(os.path.dirname(p), "model"))]
assert cands, "project dataset not found - add the mural-final dataset as input"
PROJECT = "/kaggle/working/mural_final"
if os.path.exists(PROJECT): shutil.rmtree(PROJECT)
shutil.copytree(cands[0], PROJECT)
for d in ("model", "tools", "utils", "evaluation"):
    if os.path.isdir(f"{PROJECT}/{d}"): open(f"{PROJECT}/{d}/__init__.py", "a").close()
os.chdir(PROJECT)

# ---- Attention U-Net must be at models/attention_unet.keras
target = f"{PROJECT}/models/attention_unet.keras"
if not os.path.isfile(target):
    ks = glob.glob("/kaggle/input/**/*.keras", recursive=True)
    pick = [k for k in ks if "attention" in k.lower() or "dh_mural" in k.lower()]
    assert pick, f"attention_unet.keras not found. .keras files seen: {ks}"
    os.makedirs(f"{PROJECT}/models", exist_ok=True); shutil.copy(pick[0], target)
    print("copied", pick[0], "->", target)
for f in ("attention_unet.keras", "dunhuang_inpaint_lora.safetensors", "dunhuang_canny_lora.safetensors"):
    p = f"{PROJECT}/models/{f}"
    print(f"{f:40s}", "OK" if os.path.isfile(p) else "MISSING", f"{os.path.getsize(p)/1e6:.0f} MB" if os.path.isfile(p) else "")


In [ ]:
# ---- start Streamlit in the background
env = dict(os.environ, MURAL_TF_CPU="1", MURAL_DEVICE="cuda", TF_CPP_MIN_LOG_LEVEL="2")
log = open(f"{PROJECT}/streamlit.log", "w")
st_proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true",
     "--server.enableCORS", "false", "--server.enableXsrfProtection", "false",
     "--browser.gatherUsageStats", "false", "--server.maxUploadSize", "50"],
    cwd=PROJECT, env=env, stdout=log, stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:8501/_stcore/health", timeout=2); print("Streamlit is running"); break
    except Exception: time.sleep(2)
else:
    print(open(f"{PROJECT}/streamlit.log").read()[-3000:]); raise RuntimeError("Streamlit did not start")


In [ ]:
# ---- public link through a Cloudflare tunnel (no account needed)
if not os.path.isfile("/kaggle/working/cloudflared"):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                               "/kaggle/working/cloudflared")
    os.chmod("/kaggle/working/cloudflared", 0o755)
cf_log = open("/kaggle/working/cloudflared.log", "w")
cf_proc = subprocess.Popen(["/kaggle/working/cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
                           stdout=cf_log, stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/kaggle/working/cloudflared.log").read())
    if m: url = m.group(0); break
    time.sleep(2)
print("OPEN THIS LINK:", url or "not ready - see /kaggle/working/cloudflared.log")


In [ ]:
# ---- keep-alive + live log. Leave this running while you use the app; press Stop (interrupt) to end.
try:
    while True:
        time.sleep(60)
        tail = open(f"{PROJECT}/streamlit.log").read().splitlines()[-3:]
        print(time.strftime("%H:%M:%S"), "| app alive:", st_proc.poll() is None, "|", " / ".join(tail)[-200:])
except KeyboardInterrupt:
    print("stopped watching (the app keeps running until the session ends)")


In [ ]:
# ---- run this only to shut everything down
st_proc.terminate(); cf_proc.terminate(); print("stopped")
